# Titanic Sample Project — End-to-End Classroom Solution

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
plt.rcParams['figure.figsize']=(7,5)

In [ ]:
df = pd.read_csv("/mnt/data/titanic_train.csv"); print("Shape:", df.shape); df.head()

In [ ]:
print(df.dtypes); print("\nMissing values:"); print(df.isna().sum()); print("\nTarget:"); print(df['Survived'].value_counts(normalize=True).round(3))

In [ ]:
df['Age'].dropna().hist(bins=20); plt.title('Age distribution'); plt.xlabel('Age'); plt.ylabel('Count'); plt.show()
df['Pclass'].value_counts().sort_index().plot(kind='bar'); plt.title('Passenger class counts'); plt.xlabel('Pclass'); plt.ylabel('Count'); plt.show()
df.groupby('Sex')['Survived'].mean().plot(kind='bar'); plt.title('Mean survival rate by Sex'); plt.xlabel('Sex'); plt.ylabel('Survival rate'); plt.ylim(0,1); plt.show()
df.groupby('Pclass')['Survived'].mean().plot(kind='bar'); plt.title('Mean survival rate by Pclass'); plt.xlabel('Pclass'); plt.ylabel('Survival rate'); plt.ylim(0,1); plt.show()

In [ ]:
FEATURES=['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']; TARGET='Survived'
X=df[FEATURES].copy(); y=df[TARGET].copy()
num=['Age','SibSp','Parch','Fare']; cat=['Pclass','Sex','Embarked']
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr,Xva,Ytr,Yva=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
lr=Pipeline([('prep',pre),('clf',LogisticRegression(max_iter=1000))]).fit(Xtr,Ytr)
pred_lr=lr.predict(Xva)
def evalm(y_true,y_hat):
    print("Acc",accuracy_score(y_true,y_hat),"Prec",precision_score(y_true,y_hat),"Rec",recall_score(y_true,y_hat),"F1",f1_score(y_true,y_hat))
    print(classification_report(y_true,y_hat,digits=3))
evalm(Yva,pred_lr)
cm=confusion_matrix(Yva,pred_lr); print(cm)
plt.imshow(cm,interpolation='nearest'); plt.title('Confusion Matrix — LR'); plt.colorbar(); 
plt.xticks([0,1],['No','Yes']); plt.yticks([0,1],['No','Yes']); plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

In [ ]:
rf=Pipeline([('prep',pre),('clf',RandomForestClassifier(n_estimators=300,random_state=42))]).fit(Xtr,Ytr)
pred_rf=rf.predict(Xva)
evalm(Yva,pred_rf)
cm2=confusion_matrix(Yva,pred_rf); print(cm2)
plt.imshow(cm2,interpolation='nearest'); plt.title('Confusion Matrix — RF'); plt.colorbar();
plt.xticks([0,1],['No','Yes']); plt.yticks([0,1],['No','Yes']); plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

In [ ]:
ohe=rf.named_steps['prep'].named_transformers_['cat'].named_steps['oh']
feat_names=['Age','SibSp','Parch','Fare']+list(ohe.get_feature_names_out(['Pclass','Sex','Embarked']))
r=permutation_importance(rf,Xva,Yva,n_repeats=15,random_state=42)
idx=np.argsort(r.importances_mean)[::-1]
for i in idx[:10]:
    print(feat_names[i], r.importances_mean[i], r.importances_std[i])
k=min(8,len(feat_names))
plt.bar(range(k), r.importances_mean[idx][:k]); plt.title('Permutation Importance — Top Features (RF)'); 
plt.xlabel('Feature rank'); plt.ylabel('Mean decrease in score'); plt.show()

**Takeaways:** Sex and Pclass usually dominate; RF often outperforms LR; always validate and examine errors.